# TFM Tenerife — Tarea 2.2: Aspectos específicos por reseña (PyABSA)

Extrae pares (aspecto, sentimiento) de las reseñas de TripAdvisor + Booking con PyABSA, y sube el resultado a `gold.nlp_aspectos_resenas`.

**Aviso importante:** PyABSA es una librería menos madura que `transformers` (la que usamos en la 2.1) y su formato de salida exacto puede variar entre versiones. Este notebook incluye un paso de comprobación con pocas frases antes de lanzar el proceso completo — revísalo con atención antes de seguir.

**Diseño incremental y resistente a cortes**, igual que en la 2.1: se excluye lo que ya esté en `gold`, y se guarda por bloques según se va terminando.

## Paso 0 -- Setup de Colab (Drive + secrets + GPU)

Version adaptada para Google Colab del notebook original de Guille (`analytics/tarea2/nlp_aspectos_tarea_2_2.ipynb`).

Este notebook descarga un checkpoint de PyABSA de varios cientos de MB -- la version original usaba un disco externo (`D:/...`) para no llenar `C:` y para que el checkpoint sobreviviera entre sesiones. En Colab usamos Google Drive montado con el mismo proposito: el disco local `/content/` se borra cada vez que se reinicia el runtime, Drive no.

Antes de ejecutar: icono de llave (🔑) en el panel izquierdo de Colab -> agregar los 4 secrets `AZURE_DB_HOST`, `AZURE_DB_USER`, `AZURE_DB_PASSWORD`, `AZURE_DB_NAME` (mismos valores que el `.env` del repo) y activar el acceso del notebook para cada uno.

Tambien: Runtime -> Change runtime type -> GPU. PyABSA es mas pesado que el modelo de sentimiento de la 2.1 -- en CPU funciona pero es notablemente mas lento.

La celda de abajo va a pedir autorizacion para montar tu Google Drive -- aceptala.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/tfm_tenerife/huggingface_cache', exist_ok=True)
os.makedirs('/content/drive/MyDrive/tfm_tenerife/pyabsa_trabajo', exist_ok=True)

os.environ['HF_HOME'] = '/content/drive/MyDrive/tfm_tenerife/huggingface_cache'

# PyABSA guarda sus propios checkpoints en una carpeta relativa 'checkpoints/'
# dentro del directorio de trabajo actual -- esto NO lo cubre HF_HOME. Se cambia
# el directorio de trabajo a Drive para que el checkpoint (varios cientos de MB)
# persista entre sesiones de Colab en vez de perderse al reiniciar el runtime.
os.chdir('/content/drive/MyDrive/tfm_tenerife/pyabsa_trabajo')

import torch
print('GPU disponible:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('AVISO: no hay GPU asignada en este runtime -- Runtime > Change runtime type > GPU.')
    print('PyABSA en CPU es considerablemente mas lento.')

print('HF_HOME configurado a:', os.environ['HF_HOME'])
print('Directorio de trabajo actual:', os.getcwd())


## Paso 1 — Instalar dependencias

In [ ]:
# python-dotenv no hace falta en Colab (credenciales via Colab secrets, ver Paso 0)

# NOTA: si esta celda falla con "Building wheel for tokenizers ... error" o
# "metadata-generation-failed", es un problema conocido de compatibilidad
# entre pip/setuptools y la version de Python de esta imagen de Colab -- NO
# es un error del codigo. 

%pip uninstall -y transformers tokenizers

%pip install -q 'setuptools<82'
%pip install -q pyabsa sqlalchemy psycopg2-binary pandas update_checker 'transformers>=4.44,<5'

## Paso 2 — Comprobar la versión de torch (PyABSA necesita >= 1.13)

In [ ]:
import torch

version_torch = torch.__version__
print('Version de torch instalada:', version_torch)

partes = version_torch.split('+')[0].split('.')
version_ok = (int(partes[0]), int(partes[1])) >= (1, 13)
print('Cumple >= 1.13:', version_ok)

if not version_ok:
    # Aviso reforzado (visible aunque el resto del output de la celda sea largo) --
    # el chequeo sigue sin detener la ejecucion a proposito, solo se hace mas dificil
    # de pasar por alto entre el resto de los prints/warnings de la celda.
    print()
    print('!' * 70)
    print('AVISO: la version de torch instalada NO cumple el minimo (>= 1.13) que PyABSA necesita.')
    print('Actualiza antes de seguir: pip install -U torch')
    print('!' * 70)
    print()

## Paso 3 — Conectar

In [ ]:
from google.colab import userdata
from sqlalchemy import create_engine, text
import pandas as pd

AZURE_DB_HOST = userdata.get('AZURE_DB_HOST')
AZURE_DB_USER = userdata.get('AZURE_DB_USER')
AZURE_DB_PASSWORD = userdata.get('AZURE_DB_PASSWORD')
AZURE_DB_NAME = userdata.get('AZURE_DB_NAME')
AZURE_DB_PORT = 5432  # fijo, no hay secret separado para esto (igual que en analytics/mgwr/scripts/_db.py)

db_url = f'postgresql+psycopg2://{AZURE_DB_USER}:{AZURE_DB_PASSWORD}@{AZURE_DB_HOST}:{AZURE_DB_PORT}/{AZURE_DB_NAME}'
engine = create_engine(db_url, pool_pre_ping=True, pool_recycle=280, connect_args={'sslmode': 'require'})

with engine.connect() as conn:
    version = conn.execute(text('SELECT postgis_version();')).scalar()
print('Conectado. PostGIS:', version)


## Paso 4a — Descargar recursos de spaCy (antes de cargar el extractor)

PyABSA necesita `en_core_web_sm` de spaCy para analizar la estructura gramatical. Su intento automático de descarga puede fallar dentro de Jupyter por ambigüedad de qué Python usa -- se instala aquí explícitamente, apuntando al Python del kernel actual, antes de cargar el extractor.

In [ ]:
import sys
!{sys.executable} -m spacy download en_core_web_sm

## Paso 4b — Cargar el extractor (se descarga solo, la primera vez)

In [ ]:
import update_checker
update_checker.UpdateChecker.check = lambda *args, **kwargs: None

from pyabsa import AspectTermExtraction as ATEPC

extractor = ATEPC.AspectExtractor('multilingual', auto_device=True)
print('Extractor cargado.')

## Paso 5 — Prueba con pocas frases ANTES de lanzar todo

Esto es obligatorio hacerlo antes de seguir: hay que ver la forma real de la respuesta antes de asumir nombres de campos.

In [ ]:
frases_prueba = [
    'La habitacion estaba muy limpia pero el ruido de la calle era insoportable.',
    'Precio excelente y la ubicacion perfecta, cerca de todo.',
]

resultado_prueba = extractor.predict(frases_prueba, pred_sentiment=True)
print(resultado_prueba)

**Antes de continuar:** mira el resultado impreso arriba. Debería tener, para cada frase, una lista de aspectos detectados (algo como `aspect`) y su sentimiento correspondiente (`sentiment`). Si los nombres de campo que ves son distintos a `aspect`/`sentiment`, avisa antes de seguir para ajustar el Paso 7 -- no lo ejecutes a ciegas si la forma no coincide con lo que se espera aquí.

## Paso 6 — Extraer reseñas NUEVAS de TripAdvisor + Booking

Igual que en la 2.1: se crea la tabla si no existe, y se excluye lo que ya tenga resultado. Como cada reseña puede generar varias filas (una por aspecto) o ninguna (si no se detecta ningún aspecto), se usa `DISTINCT resena_id` para saber qué reseñas ya se procesaron -- incluyendo una fila con aspecto/sentimiento en NULL para las que no dieron ningún aspecto, así no se vuelven a intentar cada vez.

In [ ]:
with engine.begin() as conn:
    conn.execute(text('''
        CREATE TABLE IF NOT EXISTS gold.nlp_aspectos_resenas (
            resena_id text,
            hotel_id text,
            aspecto text,
            sentimiento text,
            confianza double precision,
            fuente text
        )
    '''))
print('Tabla gold.nlp_aspectos_resenas lista (creada si no existia).')

consulta_resenas = '''
    SELECT 'tripadvisor' AS fuente, r.review_id::text AS resena_id, r.location_id::text AS hotel_id, r.texto AS review_text
    FROM silver.silver_tripadvisor_resenas r
    WHERE r.texto IS NOT NULL AND EXTRACT(YEAR FROM r.fecha_publicacion) > 2021
      AND NOT EXISTS (
          SELECT 1 FROM gold.nlp_aspectos_resenas g
          WHERE g.resena_id = r.review_id::text AND g.fuente = 'tripadvisor'
      )

    UNION ALL

    SELECT 'booking' AS fuente, b.review_id AS resena_id, b.establishment_id AS hotel_id, b.review_text
    FROM silver.silver_booking_reviews b
    WHERE b.review_text IS NOT NULL AND EXTRACT(YEAR FROM b.review_date) > 2021
      AND NOT EXISTS (
          SELECT 1 FROM gold.nlp_aspectos_resenas g
          WHERE g.resena_id = b.review_id AND g.fuente = 'booking'
      );
'''

df_resenas = pd.read_sql(consulta_resenas, engine)
print('Reseñas NUEVAS a analizar esta vez:', len(df_resenas))
print(df_resenas['fuente'].value_counts())

## Paso 7 — Procesar y guardar por bloques (resistente a cortes)

**Revisa la función `parsear_resultado` antes de ejecutar** -- está escrita según la forma de respuesta esperada (`aspect`, `sentiment`), pero ajústala si el Paso 5 te mostró nombres de campo distintos.

**Fragmentación de reseñas largas (ver diagnóstico en `docs/contexto_maestro_proyecto_ptna.md`, sección 6):** el checkpoint `multilingual` trunca en silencio a `max_seq_len` (128 tokens, 126 útiles) -- sin este fix, el ~21% de las reseñas (concentrado en las reseñas *largas*, las más ricas en contenido) quedaba con `aspecto = NULL` no porque no tuvieran aspectos, sino porque el contenido relevante caía fuera de la ventana visible del modelo. Antes de llamar a `predict()`, cada reseña que supera el límite se parte en fragmentos por párrafo (celda siguiente); los aspectos de todos los fragmentos de una misma reseña se combinan y deduplican en `parsear_resultado`.

In [ ]:
def _dividir_texto_largo(texto, tokenizer, limite_tokens):
    """Corta un texto por palabras (sin partir un token a la mitad) cuando un
    UNICO parrafo ya supera el limite el solo -- caso raro (reseñas sin saltos
    de linea internos), pero posible; sin este fallback, fragmentar_por_tokens
    podria devolver un fragmento que sigue superando el limite."""
    palabras = texto.split(' ')
    fragmentos = []
    actual = []
    actual_len = 0
    for palabra in palabras:
        largo_palabra = len(tokenizer.tokenize(palabra))
        if actual and actual_len + largo_palabra > limite_tokens:
            fragmentos.append(' '.join(actual))
            actual = []
            actual_len = 0
        actual.append(palabra)
        actual_len += largo_palabra
    if actual:
        fragmentos.append(' '.join(actual))
    return fragmentos


def fragmentar_por_tokens(texto, tokenizer, limite_tokens):
    """Parte `texto` en fragmentos de a lo sumo `limite_tokens` tokens reales
    (medidos con el tokenizer del extractor, no por caracteres), agrupando
    parrafos consecutivos -- mismo criterio validado en el diagnostico previo
    (ver docs/contexto_maestro_proyecto_ptna.md, seccion 6): agrupar por
    parrafo preserva mejor el contexto de cada aspecto que cortar a ciegas
    cada N caracteres.

    Si un PyABSA no llega a procesar todo el texto de una reseña porque el
    checkpoint trunca en silencio mas alla de max_seq_len, esta funcion evita
    perder los aspectos que caerian fuera de esa ventana."""
    parrafos = [p.strip() for p in texto.split('\n') if p.strip()]
    if not parrafos:
        return [texto.strip()] if texto.strip() else []

    fragmentos = []
    actual = []
    actual_len = 0
    for parrafo in parrafos:
        largo_parrafo = len(tokenizer.tokenize(parrafo))

        if largo_parrafo > limite_tokens:
            # Un parrafo individual ya supera el limite -- se cierra el
            # fragmento acumulado hasta ahora y se corta este parrafo aparte,
            # por palabras (ver _dividir_texto_largo).
            if actual:
                fragmentos.append(' '.join(actual))
                actual = []
                actual_len = 0
            fragmentos.extend(_dividir_texto_largo(parrafo, tokenizer, limite_tokens))
            continue

        if actual and actual_len + largo_parrafo > limite_tokens:
            fragmentos.append(' '.join(actual))
            actual = []
            actual_len = 0
        actual.append(parrafo)
        actual_len += largo_parrafo

    if actual:
        fragmentos.append(' '.join(actual))
    return fragmentos


# max_seq_len del checkpoint 'multilingual' cargado en el Paso 4b, menos 2
# tokens reservados para especiales ([CLS]/[SEP]) -- se calcula del config
# REAL ya cargado, no se hardcodea 126, por si algun dia se usa otro checkpoint
# con una ventana distinta.
LIMITE_TOKENS_FRAGMENTO = extractor.config.max_seq_len - 2
print(f"Limite de tokens por fragmento (max_seq_len - 2): {LIMITE_TOKENS_FRAGMENTO}")


In [ ]:
# Se emite una unica vez (no una por fila) para no inundar el output si el
# formato de PyABSA cambio de version -- ver diagnostico de analytics/tarea2/.
_campos_faltantes_avisado = False

_CAMPOS_ESPERADOS_PYABSA = ('aspect', 'sentiment', 'confidence')


def parsear_resultado(fila, resultado_pyabsa):
    """Convierte la salida de PyABSA para UNA reseña en una lista de filas
    (una por aspecto detectado). Si no se detecta ningun aspecto, devuelve
    una unica fila con aspecto/sentimiento en None, para marcar la reseña
    como procesada igualmente.

    Endurecido tras el diagnostico de analytics/tarea2/: antes, si PyABSA
    cambiaba el nombre de un campo, `.get(campo, [])` devolvia una lista vacia
    en silencio y CADA reseña quedaba marcada como 'sin aspecto detectado' sin
    ningun aviso. Ahora se loguea un warning explicito (una sola vez) si falta
    alguno de los campos esperados.

    Deduplicacion (ver Paso 7, seccion de fragmentacion): cuando una reseña se
    dividio en varios fragmentos por superar el limite de tokens del
    checkpoint, `resultado_pyabsa` llega aca ya con los aspectos de TODOS los
    fragmentos combinados -- el mismo aspecto puede repetirse si se menciono
    en mas de un fragmento (ej. "comida" al principio y al final de una
    reseña larga). Se deduplica por aspecto en minuscula, quedandose con la
    ocurrencia de MAYOR confianza: ante dos detecciones del mismo aspecto, la
    de mayor confianza es la mejor estimacion puntual disponible. No se
    intenta combinar sentimientos contradictorios entre fragmentos (ej.
    Positive en uno, Negative en otro) -- no hay una regla obviamente mejor
    que promediar o votar, y se prefirio no inventar una; se queda con el
    sentimiento de la ocurrencia de mayor confianza, sin mas."""
    global _campos_faltantes_avisado

    campos_faltantes = [c for c in _CAMPOS_ESPERADOS_PYABSA if c not in resultado_pyabsa]
    if campos_faltantes and not _campos_faltantes_avisado:
        print('!' * 70)
        print(f"AVISO: PyABSA no devolvio el/los campo(s) {campos_faltantes} en su resultado.")
        print('El formato de salida puede haber cambiado de version -- revisa el Paso 5')
        print('(prueba con pocas frases) antes de confiar en estos resultados.')
        print('Este aviso solo se muestra una vez por ejecucion.')
        print('!' * 70)
        _campos_faltantes_avisado = True

    aspectos = resultado_pyabsa.get('aspect', [])
    sentimientos = resultado_pyabsa.get('sentiment', [])
    confianzas = resultado_pyabsa.get('confidence', [])

    if not aspectos:
        return [{
            'resena_id': fila['resena_id'],
            'hotel_id': fila['hotel_id'],
            'aspecto': None,
            'sentimiento': None,
            'confianza': None,
            'fuente': fila['fuente'],
        }]

    mejor_por_aspecto = {}  # aspecto en minuscula -> (sentimiento, confianza)
    for aspecto, sentimiento, confianza in zip(aspectos, sentimientos, confianzas):
        aspecto_norm = aspecto.lower() if aspecto else aspecto
        anterior = mejor_por_aspecto.get(aspecto_norm)
        si_reemplaza = (
            anterior is None
            or (confianza is not None and (anterior[1] is None or confianza > anterior[1]))
        )
        if si_reemplaza:
            mejor_por_aspecto[aspecto_norm] = (sentimiento, confianza)

    filas = []
    for aspecto_norm, (sentimiento, confianza) in mejor_por_aspecto.items():
        filas.append({
            'resena_id': fila['resena_id'],
            'hotel_id': fila['hotel_id'],
            'aspecto': aspecto_norm,
            'sentimiento': sentimiento,
            'confianza': confianza,
            'fuente': fila['fuente'],
        })
    return filas

In [ ]:
import time

FILAS_POR_CHECKPOINT = 100  # PyABSA suele ser mas pesado por texto que el BERT de sentimiento

total = len(df_resenas)
procesadas_total = 0
inicio_general = time.time()

if total == 0:
    print('No hay resenas nuevas que procesar -- todo lo disponible ya esta en gold.')
else:
    for inicio_chunk in range(0, total, FILAS_POR_CHECKPOINT):
        chunk = df_resenas.iloc[inicio_chunk:inicio_chunk + FILAS_POR_CHECKPOINT].copy().reset_index(drop=True)

        # Fragmentar ANTES de predict() las reseñas que superan el limite de
        # tokens (ver Paso 7, celda de fragmentacion) -- se arma una lista
        # PLANA con los fragmentos de TODAS las reseñas del bloque para
        # mandarlos en un solo predict() (mas eficiente que uno por reseña),
        # guardando a que fila de `chunk` pertenece cada fragmento para poder
        # reagruparlos despues.
        fragmentos_planos = []
        fragmento_a_fila = []
        for idx_fila, fila in chunk.iterrows():
            texto = fila['review_text']
            n_tokens = len(extractor.tokenizer.tokenize(texto))
            if n_tokens > LIMITE_TOKENS_FRAGMENTO:
                partes = fragmentar_por_tokens(texto, extractor.tokenizer, LIMITE_TOKENS_FRAGMENTO)
            else:
                partes = [texto]
            for parte in partes:
                fragmentos_planos.append(parte)
                fragmento_a_fila.append(idx_fila)

        resultados_fragmentos = extractor.predict(fragmentos_planos, pred_sentiment=True)

        # Reagrupar los resultados de cada fragmento en su reseña de origen
        # ANTES de parsear -- una reseña fragmentada puede detectar el mismo
        # aspecto en mas de un fragmento; parsear_resultado se encarga de
        # deduplicar quedandose con la ocurrencia de mayor confianza (ver su
        # docstring).
        resultados_por_fila = {}
        for idx_fila, resultado in zip(fragmento_a_fila, resultados_fragmentos):
            acumulado = resultados_por_fila.setdefault(
                idx_fila, {'aspect': [], 'sentiment': [], 'confidence': []}
            )
            acumulado['aspect'].extend(resultado.get('aspect', []))
            acumulado['sentiment'].extend(resultado.get('sentiment', []))
            acumulado['confidence'].extend(resultado.get('confidence', []))

        filas_salida = []
        for idx_fila, fila in chunk.iterrows():
            resultado_combinado = resultados_por_fila.get(
                idx_fila, {'aspect': [], 'sentiment': [], 'confidence': []}
            )
            filas_salida.extend(parsear_resultado(fila, resultado_combinado))

        df_salida = pd.DataFrame(filas_salida)
        df_salida.to_sql('nlp_aspectos_resenas', engine, schema='gold', if_exists='append', index=False)

        procesadas_total += len(chunk)
        transcurrido = time.time() - inicio_general
        velocidad = procesadas_total / transcurrido if transcurrido > 0 else 0
        restantes = total - procesadas_total
        eta_min = (restantes / velocidad / 60) if velocidad > 0 else 0

        print(f'Guardadas {procesadas_total}/{total} resenas ({len(df_salida)} filas de aspectos en este bloque, '
              f'{len(fragmentos_planos)} fragmentos procesados) -- '
              f'{velocidad:.1f} resenas/seg -- estimado restante: {eta_min:.0f} min')

    with engine.begin() as conn:
        conn.execute(text(
            'CREATE INDEX IF NOT EXISTS idx_nlp_aspectos_resena_id '
            'ON gold.nlp_aspectos_resenas (resena_id)'
        ))
        conn.execute(text(
            'CREATE INDEX IF NOT EXISTS idx_nlp_aspectos_hotel_id '
            'ON gold.nlp_aspectos_resenas (hotel_id)'
        ))

    print()
    print('Completado. Total de resenas procesadas en esta ejecucion:', procesadas_total)

## Paso 8 — Verificar

In [ ]:
with engine.connect() as conn:
    resumen_fuente = pd.read_sql('''
        SELECT fuente, COUNT(DISTINCT resena_id) AS resenas, COUNT(*) AS filas_aspecto
        FROM gold.nlp_aspectos_resenas
        GROUP BY fuente
    ''', conn)
    top_aspectos = pd.read_sql('''
        SELECT aspecto, sentimiento, COUNT(*) AS total, AVG(confianza) AS confianza_media
        FROM gold.nlp_aspectos_resenas
        WHERE aspecto IS NOT NULL
        GROUP BY aspecto, sentimiento
        ORDER BY total DESC
        LIMIT 20
    ''', conn)

display(resumen_fuente)
display(top_aspectos)

## Notas

- **Confirmado contra PyABSA real** (31 agosto 2026): los nombres de campo `aspect`, `sentiment` y `confidence` son correctos tal como estaban escritos en `parsear_resultado`. Resultado de prueba real: `"habitacion"` positivo (0.99), `"ruido"` negativo (0.99) para una frase con ambos matices -- el modelo distingue bien aspectos opuestos dentro de la misma reseña.
- Se añadió la columna `confianza` (no pedida en el ticket original) tras ver que PyABSA la devuelve de serie -- útil para poder filtrar más adelante por aspectos de baja confianza si hiciera falta.
- El ticket no pedía cruce espacial para esta tarea (a diferencia de la 2.1) -- se ha añadido `hotel_id` y `fuente` igualmente, por si hace falta cruzarlo más adelante con la malla H3 u otras tablas, siguiendo el mismo criterio que en 2.1.
- La lista de aspectos del ticket (`precio, limpieza, ubicacion, transporte, naturaleza, servicio, ruido`) es orientativa, no una restricción -- el propio ticket aclara que el modelo detecta los aspectos automáticamente, así que pueden aparecer otros aspectos no listados. No se ha filtrado nada en este notebook para no perder información real.
- **Aspectos normalizados a minúsculas** antes de guardar (ej. "Precio" y "precio" se guardan igual) -- evita que el mismo aspecto cuente como dos categorías distintas solo por mayúsculas según su posición en la frase.
- **Diseño incremental y resistente a cortes**, igual que en la 2.1: si el proceso se corta a mitad, solo se pierde el bloque en curso -- vuelve a ejecutar el notebook entero y continuará donde se quedó.

---

**Nota de adaptacion a Colab:** copia adaptada del notebook original de Guille (`analytics/tarea2/nlp_aspectos_tarea_2_2.ipynb`), para correr en paralelo al trabajo del Bloque 5. El original queda intacto. Cambios: credenciales via Colab secrets, `HF_HOME` y directorio de trabajo de PyABSA movidos a Google Drive (antes `D:/huggingface_cache` y `D:/pyabsa_trabajo`, especificos de una maquina Windows con disco externo), `parsear_resultado` ahora avisa explicitamente si PyABSA no devuelve los campos esperados (`aspect`/`sentiment`/`confidence`) en vez de fallar en silencio, y el aviso de version de torch insuficiente se hizo mas visible. La logica de negocio (queries, extraccion de aspectos) no se modifico.

---

**Limpieza de entorno (14-sep-2026):** el notebook tuvo varias vueltas de troubleshooting de instalación en Colab (conflicto de `setuptools` con torch, falta de wheel precompilado para `tokenizers`, versión de `transformers` incompatible con el checkpoint de PyABSA) que habían quedado como celdas sueltas agregadas fuera de orden al principio del notebook. Se consolidaron en una única celda de instalación (Paso 1) y se reordenó el setup de Drive/GPU/rutas dentro del Paso 0. No se tocó ninguna consulta SQL, ninguna lógica de extracción o parseo de aspectos -- solo el entorno y el orden de las celdas.

---

**Fix de fragmentacion para reseñas largas (ver diagnostico en `docs/contexto_maestro_proyecto_ptna.md`, seccion 6):** se confirmo que el checkpoint `multilingual` trunca en silencio a `max_seq_len` (128 tokens, 126 utiles) sin avisar ni fallar -- probado con las 10 reseñas reales mas largas que habian quedado con `aspecto = NULL`: las 10 superaban el limite entre 8x y 22x, y las 10 daban 0 aspectos. Se probo tambien subir `max_seq_len` a 256/512 sobre el extractor ya cargado: no crashea, pero tampoco recupera aspectos -- el checkpoint no generaliza mas alla de la ventana con la que fue afinado, asi que no es un ajuste de parametro trivial. Fragmentar por parrafo y combinar SI funciono en la prueba (17 aspectos recuperados en una reseña que daba 0 sin fragmentar), por eso se implemento esa opcion (ver funciones `fragmentar_por_tokens` / `_dividir_texto_largo` antes de `parsear_resultado`, y el loop del Paso 7 actualizado). Esto solo afecta a reseñas NUEVAS a procesar -- el filtro `NOT EXISTS` del Paso 6 sigue siendo el que decide que reseñas entran a este loop, no se reprocesa nada existente.